# So sánh & Đánh giá 4 Mô hình CNN
## Bài tập 05 — Intelligent System Development

**Học viện Công nghệ Bưu chính Viễn thông (PTIT) — Khoa CNTT 1**
- **Sinh viên thực hiện:** Nguyễn Nam Hải (Mã SV: B23DCCN277 — Lớp: D23CTPM01 — Nhóm: CT01)
- **Giảng viên hướng dẫn:** PGS.TS. Trần Đình Quế
- **GitHub Repository:** [https://github.com/HandQ2212/intel-sys-assignment-05](https://github.com/HandQ2212/intel-sys-assignment-05)

### Nguồn Dữ Liệu Kaggle Chính Thức
1. **CIFAR-10 (Color 32x32 RGB):** [kaggle.com/datasets/ayush1220/cifar10](https://www.kaggle.com/datasets/ayush1220/cifar10)
2. **MNIST (400k Augmented MNIST Extended):** [kaggle.com/datasets/alexandrelemercier/400k-augmented-mnist-extended-handwritten-digits](https://www.kaggle.com/datasets/alexandrelemercier/400k-augmented-mnist-extended-handwritten-digits)
3. **Diabetes CDC (Health Indicators Dataset):** [kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset)

Notebook này tổng hợp và so sánh kết quả từ 12 thí nghiệm:
- **3 tập dữ liệu**: CIFAR-10, MNIST, Diabetes
- **4 mô hình CNN**: Basic CNN, VGG-style, ResNet-style, Attention CNN (CBAM)

### Tiêu chí đánh giá
| Tiêu chí | Mô tả |
|----------|--------|
| **Accuracy** | Tỷ lệ dự đoán đúng tổng thể |
| **F1-Score** | Trung bình điều hòa của Precision và Recall |
| **Training Loss** | Đường cong loss qua các epoch |
| **Parameters** | Tổng số tham số của mô hình |
| **Training Time** | Thời gian huấn luyện |

## Tổng quan các Mô hình

### M1: Basic CNN (Conv + BN + ReLU + Pool)
$$\hat{y} = f_{\text{FC}} \circ f_{\text{pool}_2} \circ f_{\text{relu}_2} \circ f_{\text{conv}_2} \circ f_{\text{pool}_1} \circ f_{\text{relu}_1} \circ f_{\text{conv}_1}(X)$$

### M2: VGG-style CNN (Deep 3×3 stacking)
Nhiều lớp Conv $3 \times 3$ xếp chồng → tăng depth, giảm params so với kernel lớn.

### M3: ResNet-style CNN (Residual Connections)
$$Y = F(X) + X$$
Skip connection cho phép gradient chảy trực tiếp → train được mạng sâu hơn.

### M4: Attention CNN (CBAM)
$$f_{\text{CBAM}}(X) = M_s(M_c(X) \odot X) \odot (M_c(X) \odot X)$$
Channel Attention ("WHAT?") + Spatial Attention ("WHERE?").

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Tải Kết quả

Kết quả được lưu từ mỗi notebook training vào thư mục `results/`.

In [ ]:
# Load all result files
result_files = sorted(glob.glob('results/*.json'))

if not result_files:
    print('⚠️  Chưa có kết quả! Hãy chạy các notebook training trước.')
    print('Các file cần chạy:')
    notebooks = [
        'cifar10_basic_cnn.ipynb', 'cifar10_vgg_cnn.ipynb',
        'cifar10_resnet_cnn.ipynb', 'cifar10_attention_cnn.ipynb',
        'mnist_basic_cnn.ipynb', 'mnist_vgg_cnn.ipynb',
        'mnist_resnet_cnn.ipynb', 'mnist_attention_cnn.ipynb',
        'diabetes_basic_cnn.ipynb', 'diabetes_vgg_cnn.ipynb',
        'diabetes_resnet_cnn.ipynb', 'diabetes_attention_cnn.ipynb',
    ]
    for nb in notebooks:
        print(f'  - {nb}')
else:
    all_results = []
    for f in result_files:
        with open(f, 'r') as fp:
            data = json.load(fp)
        all_results.append(data)
        print(f'✓ Loaded: {os.path.basename(f)}')
    print(f'\nTổng cộng: {len(all_results)} kết quả')

## 2. Bảng So sánh Tổng hợp

In [ ]:
if result_files:
    # Build comparison DataFrame
    rows = []
    for r in all_results:
        rows.append({
            'Dataset': r.get('dataset', 'N/A'),
            'Model': r.get('model', 'N/A'),
            'Test Accuracy': r.get('test_accs', [0])[-1] if r.get('test_accs') else 0,
            'Test Loss': r.get('test_losses', [0])[-1] if r.get('test_losses') else 0,
            'Parameters': r.get('num_params', 0),
            'Epochs': r.get('epochs', 0),
        })
    
    df_results = pd.DataFrame(rows)
    
    # Format for display
    df_display = df_results.copy()
    df_display['Test Accuracy'] = df_display['Test Accuracy'].apply(lambda x: f'{x:.4f}')
    df_display['Test Loss'] = df_display['Test Loss'].apply(lambda x: f'{x:.4f}')
    df_display['Parameters'] = df_display['Parameters'].apply(lambda x: f'{x:,}')
    
    print('═' * 80)
    print('BẢNG SO SÁNH TỔNG HỢP CÁC MÔ HÌNH CNN')
    print('═' * 80)
    display(df_display)
else:
    print('Chưa có kết quả để so sánh.')

## 3. Biểu đồ So sánh Accuracy

In [ ]:
if result_files:
    datasets = df_results['Dataset'].unique()
    models = df_results['Model'].unique()
    
    fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 5))
    if len(datasets) == 1:
        axes = [axes]
    
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    
    for idx, ds in enumerate(datasets):
        subset = df_results[df_results['Dataset'] == ds]
        bars = axes[idx].bar(subset['Model'], subset['Test Accuracy'].astype(float),
                            color=colors[:len(subset)])
        axes[idx].set_title(f'{ds}', fontsize=14, fontweight='bold')
        axes[idx].set_ylabel('Accuracy')
        axes[idx].set_ylim(0, 1.05)
        axes[idx].tick_params(axis='x', rotation=30)
        
        # Add value labels on bars
        for bar, val in zip(bars, subset['Test Accuracy'].astype(float)):
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.suptitle('So sánh Accuracy giữa các Mô hình CNN', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 📌 Phân tích Chi tiết Biểu đồ Accuracy
- **CIFAR-10**: Attention CNN (CBAM) đạt độ chính xác cao nhất (**35.20%**), vượt trội hơn ResNet (32.53%) và VGG (28.67%). Điều này khẳng định cơ chế chú ý giúp lọc nhiễu phông nền rất tốt trên ảnh độ phân giải thấp  	imes 32$.
- **MNIST**: VGG (95.25%) và ResNet (94.13%) thể hiện bước nhảy vọt so với Basic CNN (42.00%). Điều này chứng minh vai trò quyết định của độ sâu mạng trong việc trích xuất đặc trưng hình thái chữ số.
- **Diabetes CDC**: Cả 4 mô hình hội tụ đồng đều quanh mức ~80%, phản ánh tính bão hòa của kiến trúc tích chập trên dữ liệu bảng.

### 🔄 Mối quan hệ và Chuyển tiếp sang Bước tiếp theo
Biểu đồ Accuracy cho thấy kết quả cuối cùng nhưng chưa phản ánh được **động lực hội tụ (convergence dynamics)** và nguy cơ overfitting qua từng epoch. Để hiểu rõ hành vi tối ưu hóa của từng mô hình, bước tiếp theo cần đối sánh **Đường cong Training & Validation (Loss & Accuracy)** qua các epoch.

## 4. So sánh Đường cong Training

In [ ]:
if result_files:
    datasets = df_results['Dataset'].unique()
    
    fig, axes = plt.subplots(len(datasets), 2, figsize=(14, 5*len(datasets)))
    if len(datasets) == 1:
        axes = axes.reshape(1, -1)
    
    colors = {'Basic CNN': '#2196F3', 'VGG-style CNN': '#4CAF50',
              'ResNet-style CNN': '#FF9800', 'Attention CNN (CBAM)': '#E91E63',
              'Basic CNN 1D': '#2196F3', 'VGG-style CNN 1D': '#4CAF50',
              'ResNet-style CNN 1D': '#FF9800', 'Attention CNN 1D (CBAM)': '#E91E63'}
    
    for ds_idx, ds in enumerate(datasets):
        ds_results = [r for r in all_results if r.get('dataset') == ds]
        
        for r in ds_results:
            model_name = r.get('model', 'N/A')
            color = colors.get(model_name, '#333333')
            
            if r.get('train_losses'):
                axes[ds_idx, 0].plot(r['train_losses'], label=f'{model_name} (train)',
                                     color=color, linestyle='-')
            if r.get('test_losses'):
                axes[ds_idx, 0].plot(r['test_losses'], label=f'{model_name} (test)',
                                     color=color, linestyle='--')
            
            if r.get('train_accs'):
                axes[ds_idx, 1].plot(r['train_accs'], label=f'{model_name} (train)',
                                     color=color, linestyle='-')
            if r.get('test_accs'):
                axes[ds_idx, 1].plot(r['test_accs'], label=f'{model_name} (test)',
                                     color=color, linestyle='--')
        
        axes[ds_idx, 0].set_title(f'{ds} — Loss', fontsize=13)
        axes[ds_idx, 0].set_xlabel('Epoch')
        axes[ds_idx, 0].set_ylabel('Loss')
        axes[ds_idx, 0].legend(fontsize=8)
        axes[ds_idx, 0].grid(True)
        
        axes[ds_idx, 1].set_title(f'{ds} — Accuracy', fontsize=13)
        axes[ds_idx, 1].set_xlabel('Epoch')
        axes[ds_idx, 1].set_ylabel('Accuracy')
        axes[ds_idx, 1].legend(fontsize=8)
        axes[ds_idx, 1].grid(True)
    
    plt.suptitle('So sánh Đường cong Training', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 📌 Phân tích Chi tiết Biểu đồ Đường cong Training
- **Hiện tượng Overfitting ở VGG**: Trên CIFAR-10, đường train loss của VGG giảm sâu nhất nhưng test loss lại phân kỳ tăng dần từ epoch 8, cho thấy mạng 1.25M tham số bị quá khớp.
- **Độ ổn định của ResNet**: Nhờ đường truyền tắt  = F(X) + X$, đường loss của ResNet duy trì ổn định, không bị bùng nổ hay suy thoái gradient.
- **Tốc độ hội tụ của CBAM**: Attention CNN giảm test loss mượt mà nhất trên cả 3 tập dữ liệu nhờ tự động tái hiệu chuẩn các kênh đặc trưng quan trọng.

### 🔄 Mối quan hệ và Chuyển tiếp sang Bước tiếp theo
Các đường cong huấn luyện đã làm rõ nguy cơ overfitting của các mạng quá lớn. Điều này đặt ra câu hỏi về **chi phí tài nguyên phần cứng**: Mô hình nào thực sự gọn nhẹ và mô hình nào đang lãng phí tham số? Bước tiếp theo cần định lượng cụ thể **Số lượng tham số (Model Complexity)** giữa các kiến trúc.

## 5. So sánh Số lượng Tham số

In [ ]:
if result_files:
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Group by model
    labels = []
    param_counts = []
    bar_colors = []
    color_map = {'Basic': '#2196F3', 'VGG': '#4CAF50', 'ResNet': '#FF9800', 'Attention': '#E91E63'}
    
    for r in all_results:
        label = f"{r.get('dataset', '?')}\n{r.get('model', '?')}"
        labels.append(label)
        param_counts.append(r.get('num_params', 0))
        
        model_name = r.get('model', '')
        if 'Basic' in model_name:
            bar_colors.append(color_map['Basic'])
        elif 'VGG' in model_name:
            bar_colors.append(color_map['VGG'])
        elif 'ResNet' in model_name:
            bar_colors.append(color_map['ResNet'])
        elif 'Attention' in model_name:
            bar_colors.append(color_map['Attention'])
        else:
            bar_colors.append('#999999')
    
    bars = ax.bar(range(len(labels)), param_counts, color=bar_colors)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8, rotation=0)
    ax.set_ylabel('Number of Parameters')
    ax.set_title('So sánh Số lượng Tham số', fontsize=14, fontweight='bold')
    
    for bar, count in zip(bars, param_counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
               f'{count:,}', ha='center', va='bottom', fontsize=7)
    
    plt.tight_layout()
    plt.show()

### 📌 Phân tích Chi tiết Biểu đồ Số lượng Tham số
- **Basic CNN**: Cực kỳ nhỏ gọn (~6.5k - 20k params), phù hợp làm baseline nhưng năng lực biểu diễn hạn chế.
- **VGG & ResNet**: Bùng nổ tham số lên tới 1.25M - 1.86M params (gấp gần 100 lần Basic CNN) do các khối tích chập đa kênh và tầng phân loại Dense lớn.
- **Attention CNN (CBAM)**: Đạt hiệu quả phi thường khi chỉ tốn ~9k params (1D) và ~98k params (2D), nhẹ hơn 12 đến 20 lần so với VGG và ResNet.

### 🔄 Mối quan hệ và Chuyển tiếp sang Bước tiếp theo
Sự chênh lệch lớn giữa Accuracy và Số tham số chỉ ra rằng mô hình phức tạp hơn chưa chắc đã tốt hơn. Để tổng kết tương quan toàn diện giữa các tập dữ liệu và các kiến trúc mô hình trên một ma trận trực quan duy nhất, bước tiếp theo là xây dựng **Accuracy Heatmap** tổng hợp.

## 6. Heatmap So sánh

In [ ]:
if result_files:
    # Create pivot table: Dataset × Model → Accuracy
    pivot = df_results.pivot_table(
        values='Test Accuracy', 
        index='Dataset', 
        columns='Model',
        aggfunc='first'
    ).astype(float)
    
    plt.figure(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', 
                linewidths=1, linecolor='white',
                cbar_kws={'label': 'Accuracy'})
    plt.title('Accuracy Heatmap: Dataset × Model', fontsize=14, fontweight='bold')
    plt.ylabel('Dataset')
    plt.xlabel('Model')
    plt.tight_layout()
    plt.show()

### 📌 Phân tích Chi tiết Biểu đồ Heatmap
- **Hàng MNIST**: Vùng màu nóng nhất (độ chính xác >93% - 95.25% ở các mô hình phát triển), thể hiện dữ liệu ảnh nét chữ có tính cấu trúc hình học cao rất thích hợp cho CNN.
- **Hàng CIFAR-10**: Cột Attention CBAM nổi bật nhất, thể hiện tính ưu việt của cơ chế chú ý khi phân loại ảnh màu tự nhiên phức tạp.
- **Hàng Diabetes CDC**: Màu sắc đồng đều tuyệt đối quanh 0.80 trên mọi cột, khẳng định 1D CNN trên dữ liệu bảng mang tính thực nghiệm sư phạm và nhanh chóng chạm trần hiệu năng.

### 🔄 Mối quan hệ và Chuyển tiếp sang Bước tiếp theo
Ma trận nhiệt đã khép lại bức tranh đối chuẩn trực quan đa chiều. Bước tiếp theo là đi vào **Phần phân tích thống kê định lượng chi tiết, xếp hạng (Ranking)** và rút ra các **Kết luận phương pháp luận khoa học** cho toàn bộ đề tài.

## 7. Phân tích & Nhận xét

In [ ]:
if result_files:
    print('═' * 70)
    print('PHÂN TÍCH KẾT QUẢ')
    print('═' * 70)
    
    datasets = df_results['Dataset'].unique()
    
    for ds in datasets:
        subset = df_results[df_results['Dataset'] == ds].copy()
        subset['Test Accuracy'] = subset['Test Accuracy'].astype(float)
        
        best_idx = subset['Test Accuracy'].idxmax()
        best_model = subset.loc[best_idx, 'Model']
        best_acc = subset.loc[best_idx, 'Test Accuracy']
        
        print(f'\n📊 Dataset: {ds}')
        print(f'   Mô hình tốt nhất: {best_model} (Accuracy = {best_acc:.4f})')
        print(f'   Ranking:')
        ranked = subset.sort_values('Test Accuracy', ascending=False)
        for rank, (_, row) in enumerate(ranked.iterrows(), 1):
            print(f'      {rank}. {row["Model"]}: {row["Test Accuracy"]:.4f} '
                  f'({row["Parameters"]:,.0f} params)')
    
    print(f'\n{"═" * 70}')
    print('TỔNG KẾT')
    print(f'{"═" * 70}')
    print()
    print('1. Basic CNN: Mô hình đơn giản nhất, ít tham số, phù hợp làm baseline.')
    print('2. VGG-style: Tăng depth bằng 3×3 stacking → cải thiện accuracy nhưng nhiều params hơn.')
    print('3. ResNet-style: Skip connections giúp train sâu hơn, thường cho kết quả tốt.')
    print('4. Attention CNN: CBAM giúp model tập trung vào features quan trọng.')
    print()
    print('💡 Nhận xét chung:')
    print('   - Mô hình phức tạp hơn KHÔNG luôn cho accuracy cao hơn.')
    print('   - Cần cân bằng giữa accuracy và computational cost (params).')
    print('   - Hiệu quả phụ thuộc vào đặc điểm dữ liệu:')
    print('     + Image data: CNN 2D phù hợp tự nhiên.')
    print('     + Tabular data: CNN 1D là "teaching experiment", không tối ưu.')

## 8. Kết luận

### Các bài học chính

1. **CNN là hợp hàm**: Mọi kiến trúc CNN đều có thể biểu diễn dưới dạng $\hat{y} = f_L \circ f_{L-1} \circ \cdots \circ f_1(X)$

2. **Tiến hóa kiến trúc = Giải quyết hạn chế**:
   - Basic CNN → VGG: Tăng depth bằng small kernels
   - VGG → ResNet: Skip connections giải quyết degradation
   - ResNet → CBAM: Attention chọn lọc features quan trọng

3. **Không có mô hình "tốt nhất" cho mọi bài toán**: Hiệu quả phụ thuộc vào data, task, và resources.

4. **Tabular data + CNN = Teaching experiment**: CNN được thiết kế cho dữ liệu có cấu trúc không gian (ảnh, audio). Dữ liệu tabular không có spatial locality tự nhiên.

### Công thức kiến trúc tổng quát
$$\text{new\_architecture} = \text{old\_architecture} + \text{mechanism\_addressing\_a\_limitation}$$